# Milestone 2 lab, notebook 5 of 8: Pull or push

## Where we are

| Notebook | What you do |
|---|---|
| 1. Ask only for what changed | Read the orders that changed, not all of them |
| 2. Remember where you stopped | Keep a watermark. See what happens when it moves too early |
| 3. When the API says no | Meet the rate limit and a server error. Wait, and try again |
| 4. Land it raw | Put every order in Postgres, exactly as it arrived |
| **5. Pull or push** (you are here) | A short lab: let the order system send the changes to you |
| 6. What you built has a name | Change data capture, and what this kind cannot see |
| 7. Counted twice | Run a few mornings. Watch the report go wrong |
| 8. The production version | The same logic as a tested program |

**This notebook is a demonstration.** The pipeline you are building asks the order system for data. Here you try the opposite for a few minutes, see how it behaves, and go back.

In [1]:
import json
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer

from lab_helpers import api, read_orders, set_morning

## Step 1: Who starts the transfer?

So far, your code starts every transfer. It calls the API and asks: what changed? That is a **pull**.

The opposite is a **push**: the source starts. It sends each change to you when it happens. For an API, a push usually arrives as a **webhook**: a message the source sends to your web address.

To receive a push, you need something that listens. Here is a tiny web server, in a few lines. It keeps every message it gets in a list.

In [2]:
received = []

class Receiver(BaseHTTPRequestHandler):
    def do_POST(self):
        size = int(self.headers["Content-Length"])
        received.append(json.loads(self.rfile.read(size)))
        self.send_response(200)
        self.end_headers()

    def log_message(self, *args):       # keep the notebook quiet
        pass

def start_receiver():
    server = HTTPServer(("127.0.0.1", 8011), Receiver)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server

server = start_receiver()
print("Listening on http://127.0.0.1:8011")

Listening on http://127.0.0.1:8011


## Step 2: Ask the order system to push

Tell the order system your address. Then let one day pass.

In [3]:
set_morning("2026-06-08")
api.post("/webhooks", json={"url": "http://127.0.0.1:8011/hook"})

clock = api.post("/admin/clock", json={"advance_days": 1}).json()      # Monday 07:00 -> Tuesday 07:00
print("The order system pushed:", clock["webhooks"])
print("We received:            ", len(received), "messages")

It is now Monday 2026-06-08 07:00:00 in Arden.


The order system pushed: {'sent': 88, 'failed': 0}
We received:             88 messages


You did not ask for anything. The changes came to you. Look at the first few.

In [4]:
for change in received[:4]:
    order = change["after"]
    print(f"pos {change['pos']}  op {change['op']}  order {change['order_id']}  at {change['changed_at']}  -> {order['status']}")

pos 895  op c  order 1227  at 2026-06-08T12:24:54Z  -> placed
pos 896  op u  order 1225  at 2026-06-08T12:55:00Z  -> packed
pos 897  op u  order 1227  at 2026-06-08T12:58:54Z  -> packed
pos 898  op u  order 1225  at 2026-06-08T13:28:00Z  -> out_for_delivery


## Step 3: Compare with the pull

Pull the same day, as your pipeline does.

In [5]:
pulled, calls = read_orders(updated_since="2026-06-08T12:00:00Z")

print(f"Pull: {len(pulled)} orders, in {calls} calls that we started.")
print(f"Push: {len(received)} messages about {len({c['order_id'] for c in received})} orders, in 0 calls from us.")

Pull: 29 orders, in 2 calls that we started.
Push: 88 messages about 29 orders, in 0 calls from us.


The push sent **more** than the pull found. Follow one order to see why.

In [6]:
one = max({c["order_id"] for c in received}, key=lambda i: sum(c["order_id"] == i for c in received))

print(f"Order {one}, by push: every step")
for change in received:
    if change["order_id"] == one:
        print(f"   {change['changed_at']}  {change['after']['status']}")

print(f"\nOrder {one}, by pull: one row")
for order in pulled:
    if order["order_id"] == one:
        print(f"   {order['updated_at']}  {order['status']}")

Order 1227, by push: every step
   2026-06-08T12:24:54Z  placed
   2026-06-08T12:58:54Z  packed
   2026-06-08T13:50:54Z  out_for_delivery
   2026-06-08T14:32:54Z  delivered

Order 1227, by pull: one row
   2026-06-08T14:32:54Z  delivered


The push told you about each step as it happened. The pull only saw where the order ended up.

## Step 4: The receiver goes down

Now the weak point. Stop your receiver, as if your machine crashed. Let another day pass.

In [7]:
server.shutdown()
server.server_close()
before = len(received)

clock = api.post("/admin/clock", json={"advance_days": 1}).json()      # Tuesday -> Wednesday
print("The order system pushed:", clock["webhooks"])
print("We received:            ", len(received) - before, "messages")

The order system pushed: {'sent': 0, 'failed': 140}
We received:             0 messages


The order system tried, nobody answered, and it moved on. This one does not try again.

Start the receiver again, and let a third day pass.

In [8]:
server = start_receiver()

clock = api.post("/admin/clock", json={"advance_days": 1}).json()      # Wednesday -> Thursday
new = received[before:]
print("The order system pushed:", clock["webhooks"])
print("Oldest change we now hold after the gap:", min(c["changed_at"] for c in new))

The order system pushed: {'sent': 119, 'failed': 0}
Oldest change we now hold after the gap: 2026-06-10T12:00:01Z


Tuesday's changes are gone. **Nothing failed on your side.** Your receiver simply has a hole in it, and nothing tells you.

A pull does not have this problem. It asks "what changed since my watermark?", so after a bad day it asks for two days.

In [9]:
caught_up, _ = read_orders(updated_since="2026-06-09T12:00:00Z")       # the watermark from before the gap
missing = {o["order_id"] for o in caught_up} - {c["order_id"] for c in new}
print(f"A pull from the old watermark finds {len(caught_up)} orders.")
print(f"{len(missing)} of them changed during the gap and never reached the receiver.")

A pull from the old watermark finds 98 orders.
36 of them changed during the gap and never reached the receiver.


In [10]:
api.delete("/webhooks")
server.shutdown()
server.server_close()
print("Lab cleaned up.")

Lab cleaned up.


## Pull and push, side by side

| | Pull (your pipeline) | Push (this lab) |
|---|---|---|
| Who starts the transfer | Your job. It asks | The source. It sends |
| When | On your schedule | When something happens |
| What you get | The latest state of each changed row | Every change |
| If the source is down | Your call fails **loudly** | Nothing arrives |
| If **you** are down | You catch up from the watermark | Changes can be lost **silently** |
| What you must add | A watermark | A check that data arrived |

> Silence is also a failure. For every push source, write down by when data must arrive, and alert when it does not.

There is a name for asking again and again "what is new?": **polling**. Your daily pull is polling, once a day.

## New words

| Word | Meaning | At Arden |
|---|---|---|
| **Pull** | "In a pull, your job starts the transfer. It asks the source for data on a schedule." | Your pipeline calls `GET /orders` every morning |
| **Push** | "In a push, the source starts it. It sends data when it is ready." | The webhook in this lab |
| **Polling** | "Polling means you ask again and again: what is new?" | The daily pull with a watermark |

The definitions are from the D8Loop chapters [Data Sources & Ingestion Patterns](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-sources-and-ingestion-patterns) and [Change Data Capture (CDC)](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/change-data-capture-cdc).

## What you now know

- Pull or push is the question "who starts?".
- A push brings every change with no calls from you. If you are down, it can lose changes with no error.
- A pull with a watermark catches up by itself after a bad day.
- This project stays with pull. A morning report does not need changes within seconds.

**Say it in an interview:** "I pull with a watermark. I tried a webhook in a lab: it delivered every change at once, but when my receiver was down for a day those changes were lost and nothing failed. With a pull, the next run just asks from the old watermark. If I had to use a push source, I would add a check that data arrived on time."

Next: [notebook 6](06_what_you_built_has_a_name.ipynb) gives a name to the kind of load you built.